# Task 6 — Music Genre Classification

**Goal:** classify GTZAN recordings into music genres and compare (1) track-level Librosa tabular features with Scikit-learn and (2) complete-track log-Mel spectrograms with a CNN.

This notebook is designed for Google Colab. Run top-to-bottom. The audio dataset and generated caches/models stay in the Colab runtime and are not committed to Git. Both approaches use the same stratified recording-level split; no track contributes data to more than one split.

## 1. Runtime setup

The notebook clones the project working branch, installs its package and dependencies, and records the runtime. For the CNN, select **Runtime → Change runtime type → T4 GPU** if available. The tabular experiment can run on CPU.

In [ ]:
import os, sys, json, random, platform, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Desire-in-tech/music-genre-classification.git"
BRANCH = "task-06/music-genre-classification"
PROJECT_DIR = Path("/content/music-genre-classification")

if not (PROJECT_DIR / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
%cd /content/music-genre-classification
%pip -q install -r requirements.txt
%pip -q install -e . kagglehub tqdm

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
import numpy as np
np.random.seed(SEED)

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("GPU devices:", subprocess.run(["bash","-lc","nvidia-smi"],capture_output=True,text=True).stdout[:1200] or "No NVIDIA GPU detected")


## 2. Download and locate GTZAN

The recommended Kaggle collection is downloaded with KaggleHub. If Kaggle requires authentication, create a Kaggle API token in your Kaggle account, upload `kaggle.json` when prompted, and authenticate in the next cell. Do not commit the token.

In [ ]:
import kagglehub
from music_genre_classification.data import discover_tracks

try:
    DATA_ROOT = Path(kagglehub.dataset_download("carlthome/gtzan-genre-collection"))
    print("Dataset downloaded to:", DATA_ROOT)
except Exception as exc:
    print("KaggleHub download did not complete:", repr(exc))
    print("If authentication is required, upload kaggle.json from your Kaggle account.")
    from google.colab import files
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise RuntimeError("Please upload your Kaggle API token file named kaggle.json.") from exc
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    token_path = kaggle_dir / "kaggle.json"
    token_path.write_bytes(uploaded["kaggle.json"])
    token_path.chmod(0o600)
    DATA_ROOT = Path(kagglehub.dataset_download("carlthome/gtzan-genre-collection"))

tracks = discover_tracks(DATA_ROOT)
print("Discovered recordings:", len(tracks))
print("Genres:", sorted(tracks.genre.unique()))
display(tracks.groupby("genre").size().rename("recordings").to_frame())
display(tracks.head())


## 3. Dataset quality audit

Inspect duration, sample rate, channels, unreadable files and exact byte-level duplicates. A matching SHA-256 hash identifies identical files only; it does not detect alternate encodings, near-duplicates, or artist overlap. Review the audit before interpreting scores.

In [ ]:
from music_genre_classification.data import audit_tracks

audit, exact_duplicates = audit_tracks(tracks, hash_files=True)
display(audit.groupby("genre").agg(
    recordings=("track_id","count"),
    readable=("readable","sum"),
    median_duration_s=("duration_seconds","median"),
    sample_rates=("sample_rate",lambda x: ", ".join(map(str,sorted(x.dropna().unique()))))
))
print("Unreadable recordings:", int((~audit.readable).sum()))
print("Exact duplicate files:", len(exact_duplicates))
if len(exact_duplicates):
    display(exact_duplicates[["track_id","sha256"]])
if (~audit.readable).any():
    display(audit.loc[~audit.readable,["track_id","path","error"]])
    raise RuntimeError("Resolve unreadable audio files before training; do not silently drop them.")


## 4. Fixed recording-level train/validation/test split

Split original recordings before extracting either representation. The test set remains untouched during model selection. The split manifest is saved in the runtime so the two approaches can be compared fairly.

In [ ]:
from music_genre_classification.data import split_tracks

train_tracks, val_tracks, test_tracks = split_tracks(
    tracks, test_size=0.20, validation_size=0.20, random_seed=SEED
)
split_frames = {"train":train_tracks,"validation":val_tracks,"test":test_tracks}
split_ids = {name:set(frame.track_id) for name,frame in split_frames.items()}
assert not (split_ids["train"] & split_ids["validation"])
assert not (split_ids["train"] & split_ids["test"])
assert not (split_ids["validation"] & split_ids["test"])
assert set.union(*split_ids.values()) == set(tracks.track_id)
for name, frame in split_frames.items():
    print(f"{name}: {len(frame)} recordings")
    display(frame.groupby("genre").size().rename("count").to_frame())
Path("artifacts").mkdir(exist_ok=True)
pd.concat([frame.assign(split=name) for name,frame in split_frames.items()]).to_csv("artifacts/split_manifest.csv",index=False)


## 5. Tabular feature extraction

Extract one feature vector per complete recording: MFCCs, chroma, spectral centroid/bandwidth/rolloff, zero-crossing rate, RMS energy and tempo. This step is the most time-consuming part of the tabular pipeline, so its Parquet cache is reused on subsequent runs when its track IDs match the current dataset.

In [ ]:
from music_genre_classification.features import extract_track_features, save_feature_table

FEATURE_CACHE = Path("cache/gtzan_track_features.parquet")
FEATURE_CACHE.parent.mkdir(exist_ok=True)
expected_ids = set(tracks.track_id)
if FEATURE_CACHE.exists():
    cached = pd.read_parquet(FEATURE_CACHE)
    if set(cached.track_id) == expected_ids and len(cached) == len(tracks):
        feature_table = cached
        print("Loaded cached features:", feature_table.shape)
    else:
        print("Feature cache does not match this dataset; rebuilding.")
        FEATURE_CACHE.unlink()
        feature_table = None
else:
    feature_table = None

if feature_table is None:
    feature_rows = []
    for record in tqdm(tracks.itertuples(index=False), total=len(tracks), desc="Extracting track features"):
        values = extract_track_features(record.path, sample_rate=22050, n_mfcc=20, duration=30)
        feature_rows.append({"track_id":record.track_id,"genre":record.genre,**values})
    feature_table = pd.DataFrame(feature_rows)
    save_feature_table(feature_table, FEATURE_CACHE)
assert feature_table.track_id.is_unique
assert set(feature_table.track_id) == expected_ids
print("Feature matrix:", feature_table.shape)
display(feature_table.head())


## 6. Train and select tabular baselines

Compare RBF-SVM, Random Forest and Extra Trees. Imputation and scaling (where applicable) are fitted inside each Scikit-learn pipeline using training data only. Select by validation macro-F1; only the selected model is then refitted on train + validation and evaluated on the test recordings.

In [ ]:
from music_genre_classification.models import build_baseline_models
from sklearn.metrics import f1_score
from music_genre_classification.evaluation import evaluate_predictions
import joblib

indexed_features = feature_table.set_index("track_id")
def split_xy(frame):
    ordered = indexed_features.loc[frame.track_id]
    X = ordered.drop(columns=["genre"])
    y = ordered.genre.to_numpy()
    return X,y

X_train,y_train = split_xy(train_tracks)
X_val,y_val = split_xy(val_tracks)
X_test,y_test = split_xy(test_tracks)

validation_rows = []
fitted_candidates = {}
for model_name, estimator in build_baseline_models(random_seed=SEED).items():
    estimator.fit(X_train,y_train)
    val_pred = estimator.predict(X_val)
    score = f1_score(y_val,val_pred,average="macro")
    validation_rows.append({"model":model_name,"validation_macro_f1":score})
    fitted_candidates[model_name] = estimator
tabular_selection = pd.DataFrame(validation_rows).sort_values("validation_macro_f1",ascending=False)
display(tabular_selection)
best_tabular_name = tabular_selection.iloc[0].model
print("Selected tabular model:",best_tabular_name)

X_trainval = pd.concat([X_train,X_val])
y_trainval = np.concatenate([y_train,y_val])
best_tabular = build_baseline_models(random_seed=SEED)[best_tabular_name]
best_tabular.fit(X_trainval,y_trainval)
tabular_test_pred = best_tabular.predict(X_test)
genre_labels = sorted(tracks.genre.unique())
tabular_metrics = evaluate_predictions(y_test,tabular_test_pred,genre_labels)
print("Tabular test accuracy:",round(tabular_metrics["accuracy"],4))
print("Tabular test macro-F1:",round(tabular_metrics["macro_f1"],4))
joblib.dump(best_tabular,"artifacts/best_tabular_model.joblib")


## 7. Build complete-track log-Mel spectrograms

Each recording becomes one fixed-size 128 × 256 log-Mel image. Images are cached locally. Crucially, we split by recording IDs first and then select complete images for each split—no patches from one song can leak across train/test.

In [ ]:
from music_genre_classification.features import extract_log_mel

MEL_CACHE = Path("cache/gtzan_logmel_v1.npz")
if MEL_CACHE.exists():
    archive = np.load(MEL_CACHE,allow_pickle=False)
    cached_ids = archive["track_ids"].astype(str)
    if set(cached_ids) == expected_ids and len(cached_ids) == len(tracks):
        mel_images = archive["images"]
        mel_track_ids = cached_ids
        print("Loaded spectrogram cache:",mel_images.shape)
    else:
        print("Spectrogram cache does not match dataset; rebuilding.")
        MEL_CACHE.unlink()
        mel_images = None
else:
    mel_images = None

if mel_images is None:
    mel_images = np.stack([
        extract_log_mel(path,sample_rate=22050,n_mels=128,hop_length=1024,duration=30,target_frames=256)
        for path in tqdm(tracks.path,desc="Creating log-Mel images")
    ])
    mel_track_ids = tracks.track_id.to_numpy(dtype=str)
    np.savez_compressed(MEL_CACHE,images=mel_images,track_ids=mel_track_ids)
print("Spectrogram tensor:",mel_images.shape,mel_images.dtype)
assert len(mel_track_ids)==len(tracks)


## 8. Train the spectrogram CNN

A compact CNN is trained on the same train/validation/test recording IDs. Early stopping restores the best validation-loss weights. TensorFlow is already available in standard Colab runtimes; if it is missing, install a compatible TensorFlow build and rerun this section.

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from music_genre_classification.models import build_spectrogram_cnn

tf.keras.utils.set_random_seed(SEED)
id_to_row = {track_id:i for i,track_id in enumerate(mel_track_ids)}
def mel_xy(frame):
    indices = np.array([id_to_row[x] for x in frame.track_id])
    return mel_images[indices], frame.genre.to_numpy()

X_mel_train,y_mel_train = mel_xy(train_tracks)
X_mel_val,y_mel_val = mel_xy(val_tracks)
X_mel_test,y_mel_test = mel_xy(test_tracks)
label_encoder = LabelEncoder().fit(tracks.genre)
y_mel_train_enc = label_encoder.transform(y_mel_train)
y_mel_val_enc = label_encoder.transform(y_mel_val)
y_mel_test_enc = label_encoder.transform(y_mel_test)

cnn = build_spectrogram_cnn(X_mel_train.shape[1:],len(label_encoder.classes_),random_seed=SEED)
cnn.summary()
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=2,min_lr=1e-5),
]
history = cnn.fit(
    X_mel_train,y_mel_train_enc,
    validation_data=(X_mel_val,y_mel_val_enc),
    epochs=30,batch_size=16,callbacks=callbacks,verbose=1
)


## 9. Evaluate CNN on the held-out test recordings

The test set is used only now, after training and validation-based early stopping. Both model families are evaluated against the same ordered test tracks and genre labels.

In [ ]:
cnn_probabilities = cnn.predict(X_mel_test,verbose=0)
cnn_test_pred = label_encoder.inverse_transform(cnn_probabilities.argmax(axis=1))
cnn_metrics = evaluate_predictions(y_mel_test,cnn_test_pred,genre_labels)
print("CNN test accuracy:",round(cnn_metrics["accuracy"],4))
print("CNN test macro-F1:",round(cnn_metrics["macro_f1"],4))
cnn.save("artifacts/logmel_cnn.keras")


## 10. Compare models and inspect errors

The table compares test-set metrics for the selected tabular baseline and CNN. Review per-genre scores and confusion matrices rather than relying on accuracy alone. GTZAN is small and has known quality/duplication concerns; this benchmark does not establish generalisation to arbitrary modern music.

In [ ]:
comparison = pd.DataFrame([
    {"approach":f"Tabular ({best_tabular_name})","test_accuracy":tabular_metrics["accuracy"],"test_macro_f1":tabular_metrics["macro_f1"],"test_macro_precision":tabular_metrics["macro_precision"],"test_macro_recall":tabular_metrics["macro_recall"]},
    {"approach":"Log-Mel CNN","test_accuracy":cnn_metrics["accuracy"],"test_macro_f1":cnn_metrics["macro_f1"],"test_macro_precision":cnn_metrics["macro_precision"],"test_macro_recall":cnn_metrics["macro_recall"]},
]).sort_values("test_macro_f1",ascending=False)
display(comparison)
fig,axes=plt.subplots(1,2,figsize=(16,6))
for ax,metrics,title in zip(axes,[tabular_metrics,cnn_metrics],["Tabular baseline","Log-Mel CNN"]):
    matrix=np.asarray(metrics["confusion_matrix"])
    image=ax.imshow(matrix,cmap="Blues")
    ax.set_title(title); ax.set_xlabel("Predicted genre"); ax.set_ylabel("True genre")
    ax.set_xticks(range(len(genre_labels)),genre_labels,rotation=45,ha="right")
    ax.set_yticks(range(len(genre_labels)),genre_labels)
    fig.colorbar(image,ax=ax,fraction=.046,pad=.04)
fig.tight_layout(); plt.show()

display(pd.DataFrame(tabular_metrics["classification_report"]).T)
display(pd.DataFrame(cnn_metrics["classification_report"]).T)


## 11. Misclassified recordings, learning curves and saved results

Use the errors to identify genre confusions. Save the comparison and metric payloads as JSON for reproducibility. These outputs remain in the Colab runtime until downloaded or saved elsewhere.

In [ ]:
error_table = pd.DataFrame({
    "track_id":test_tracks.track_id.to_numpy(),
    "true_genre":y_test,
    "tabular_prediction":tabular_test_pred,
    "cnn_prediction":cnn_test_pred,
})
display(error_table.loc[
    (error_table.true_genre!=error_table.tabular_prediction) |
    (error_table.true_genre!=error_table.cnn_prediction)
].head(30))

pd.DataFrame(history.history)[["loss","val_loss","accuracy","val_accuracy"]].plot(figsize=(9,4),title="CNN training history")
plt.xlabel("Epoch"); plt.grid(True); plt.show()

results = {
    "seed":SEED,
    "dataset_recordings":int(len(tracks)),
    "genre_counts":{str(k):int(v) for k,v in tracks.genre.value_counts().items()},
    "split_counts":{k:int(len(v)) for k,v in split_frames.items()},
    "tabular_model":best_tabular_name,
    "tabular_metrics":tabular_metrics,
    "cnn_metrics":cnn_metrics,
    "comparison":comparison.to_dict(orient="records"),
    "notes":["Track-level stratified split shared by both approaches.","Test set held out until final evaluation.","Exact SHA-256 duplicates audited; near-duplicates and artist leakage are not ruled out."]
}
with open("artifacts/results.json","w") as f:
    json.dump(results,f,indent=2)
comparison.to_csv("artifacts/model_comparison.csv",index=False)
error_table.to_csv("artifacts/test_predictions.csv",index=False)
print("Saved artifacts:",sorted(p.name for p in Path("artifacts").iterdir()))


## 12. Conclusions and limitations

Complete this section after reviewing the actual test metrics and error patterns. Discuss which approach performed better on macro-F1, which genres were confused, training/runtime trade-offs, and the limitations of GTZAN (small dataset, known duplicates/label issues, and possible artist overlap). Do not claim that one approach is universally superior from this single benchmark.

**Transfer learning is intentionally not included in the required run.** It is an optional extension only if the CNN baseline is stable and there is a clear benefit worth the additional compute and complexity.